# 04 — Equal-operating-point detector comparison on SYNTHETIC anomalies

**Production Pulse — Phase 5 (comparison).** Compares the **rule check**, the **PCA reconstruction-error baseline** and the **saved autoencoder** on the *same* evaluation rows and labels, with every threshold set for the *same* target false-positive rate on the *calibration-role* runs only.

What this notebook does **not** do: it does not train or modify the model (it loads `artifacts/models/autoencoder_v1`), it never loads the holdout runs, and it does not change any anomaly magnitude.

What the numbers mean: every anomaly here is **synthetic** — injected by the documented generator into fault-free TEP data. They show how well each detector finds *these* injected patterns. They are **not** evidence that real process faults would be detected.

Design (all run-disjoint, seeded):
* **fit**: training-file `train` split (scaler, rule limits, PCA, generator statistics; the autoencoder was fitted on it earlier)
* **calibrate**: testing-file `calibration` runs → one threshold per detector at `TARGET_FPR`
* **evaluate**: **all** testing-file `evaluation` runs (no sub-sampling unless `EVAL_RUNS_LIMIT` is set), anomalies injected at `ANOMALY_RATE` of rows per family
* **holdout**: never loaded (`include_holdout=False`)

Uncertainty: percentile intervals from a bootstrap over whole **runs** (rows of one run are strongly autocorrelated). PR-AUC must be read against the random-detector reference, which equals the anomaly prevalence.

> Results of the historical executed notebooks (`reports/executed/0[123]_*.executed.ipynb`) came from other selections (notebook 02 originally drew runs from all roles; notebook 03 used a 60-run random subset). They are not results of this notebook.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_here, *_here.parents)
                     if (p / "pyproject.toml").is_file() and (p / "src" / "production_pulse").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the project folder.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from production_pulse import get_paths, load_config, setup_logging
from production_pulse.workflow import prepare_fault_free_data
from production_pulse.autoencoder import AutoencoderDetector, verify_manifest
from production_pulse.baselines import PCAReconstructionDetector, SingleFeatureThresholdDetector
from production_pulse.anomaly_generation import FAMILIES, AnomalyGenerator, AnomalySpec
from production_pulse.comparison import cluster_bootstrap, paired_difference, prevalence, rate_by_sample_band
from production_pulse.evaluation import compare_detectors, threshold_for_fpr
from production_pulse.splitting import HOLDOUT_RUNS_PREVIOUSLY_SCORED

paths = get_paths()
config = load_config(root=paths.root)
logger = setup_logging(config=config, paths=paths)

# ---- settings (change here, then Run All) ----
SEED = config.random_seed                    # 42 by default
MODEL_DIR = "artifacts/models/autoencoder_v1"
ANOMALY_RATE = 0.05                          # fraction of evaluation rows made anomalous, per family (same as notebooks 02/03)
TARGET_FPR = config.evaluation.target_fpr    # the SAME target for every calibrated detector
EVAL_RUNS_LIMIT = None                       # None = ALL evaluation-role runs; an integer = deterministic random subset
N_BOOT = 100                                 # bootstrap resamples of whole runs (percentile 95 % intervals)
SAMPLE_BAND_EDGES = [1, 251, 501, 751, 960]  # training runs have 500 samples, testing runs 960
WRITE_REPORT = True                          # writes NEW files only (never evaluation_metrics.json)
REPORT_TARGET = "reports/evaluation_equal_fpr.json"
OVERWRITE = False                            # True replaces the files written by an earlier run of THIS notebook

# Never fail at the very end (after minutes of compute) because an earlier run already wrote the report files,
# and never overwrite them silently: if they exist, this run writes to time-stamped NEW names instead.
import datetime as _dt
_existing = [PROJECT_ROOT / REPORT_TARGET] + [PROJECT_ROOT / "reports" / f"equal_fpr_{n}.csv"
                                              for n in ("comparison", "bootstrap", "differences")]
RUN_SUFFIX = ""
if WRITE_REPORT and not OVERWRITE and any(p.exists() for p in _existing):
    RUN_SUFFIX = ".rerun-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    REPORT_TARGET = REPORT_TARGET.replace(".json", RUN_SUFFIX + ".json")
    print("Report files from an earlier run exist; this run will write NEW files with suffix", RUN_SUFFIX)
print("Project root:", paths.root, "| seed:", SEED, "| target FPR:", TARGET_FPR)

## 1. Data roles and the saved model (nothing is trained here)

In [ ]:
data = prepare_fault_free_data(paths, config)          # include_holdout defaults to False
assert "holdout" not in data.features, "holdout must stay locked"
print(data.summary().to_string())

ae = AutoencoderDetector.load(MODEL_DIR, paths=paths)
print("\nartifact manifest:", verify_manifest(paths.root / MODEL_DIR),
      "(absent = artifact saved before manifests; create one with scripts/write_artifact_manifest.py)")

# 1) Is the scaler stored inside the saved model the one the workflow fits on the training split?
saved_scaler, workflow_scaler = ae.scaler, data.scaler
same_cols = tuple(saved_scaler.feature_columns) == tuple(workflow_scaler.feature_columns)
d_center = float(np.max(np.abs(saved_scaler.center_ - workflow_scaler.center_)))
d_scale = float(np.max(np.abs(saved_scaler.scale_ - workflow_scaler.scale_)))
print(f"saved scaler == workflow scaler: columns {same_cols}, max|centre diff| {d_center:.3g}, max|scale diff| {d_scale:.3g}")

# 2) Does the saved threshold equal a fresh calibration on the calibration runs?
fresh = threshold_for_fpr(ae.score(data["calibration"]), TARGET_FPR)
print(f"saved AE threshold {ae.threshold_:.10f} | recomputed on calibration runs {fresh:.10f} | diff {abs(fresh - ae.threshold_):.3g}")
print("AE threshold source:", ae.threshold_source_)

## 2. Detectors and thresholds
Fitted on the training `train` split only. The calibrated detectors use the **calibration-role** runs and the same `TARGET_FPR`. The rule check is shown twice: with its fixed limits (score ≥ 1 means outside the training 0.05 %–99.95 % limits), and recalibrated to `TARGET_FPR` so it can be compared fairly. The autoencoder keeps its saved threshold (calibrated at the same target on the same calibration runs; checked above).

In [ ]:
train, calibration = data["train"], data["calibration"]
rule_fixed = SingleFeatureThresholdDetector().fit(train)                 # default fixed threshold 1.0
rule_equal = SingleFeatureThresholdDetector().fit(train)
rule_equal.calibrate(calibration, target_fpr=TARGET_FPR)
pca = PCAReconstructionDetector(n_components=0.90).fit(train)
pca.calibrate(calibration, target_fpr=TARGET_FPR)

DETECTORS = {"rule_check_fixed": rule_fixed, "rule_check_equal_fpr": rule_equal, "pca": pca, "autoencoder": ae}
thresholds = pd.DataFrame({n: {"threshold": d.threshold_, "source": d.threshold_source_} for n, d in DETECTORS.items()}).T
print("PCA components kept:", pca.n_components_, f"({pca.explained_variance_ratio_.sum():.3f} of training variance)")
thresholds

## 3. Evaluation rows
All evaluation-role runs (or a deterministic subset if `EVAL_RUNS_LIMIT` is set). Calibration and holdout runs cannot appear here: the evaluation role is built by the seeded run split.

In [ ]:
eval_x, eval_meta = data["evaluation"], data.metadata["evaluation"]
if EVAL_RUNS_LIMIT is not None:
    keep_runs = np.sort(np.random.default_rng(SEED).choice(eval_meta["simulationRun"].unique(), EVAL_RUNS_LIMIT, replace=False))
    keep = eval_meta["simulationRun"].isin(keep_runs).to_numpy()
    eval_x, eval_meta = eval_x[keep], eval_meta[keep]
runs = eval_meta["simulationRun"].to_numpy()
eval_runs = sorted(set(runs.tolist()))
assert not set(eval_runs) & set(data.test_split.runs("calibration")) and not set(eval_runs) & set(data.test_split.runs("holdout"))
assert not set(eval_runs) & set(HOLDOUT_RUNS_PREVIOUSLY_SCORED)
print(f"evaluation rows {len(eval_x):,} from {len(eval_runs)} runs (of {len(data.test_split.runs('evaluation'))} eligible); "
      f"samples per run {eval_meta.groupby('simulationRun').size().unique().tolist()}")

## 4. Clean false-alarm rate (no injection) — the operating points really are comparable?

In [ ]:
clean = {n: d.detect(eval_x) for n, d in DETECTORS.items()}          # also refuses rows seen in fitting/calibration
clean_rows = []
for n, det in clean.items():
    per_run = pd.Series(det.predictions).groupby(runs).mean()
    clean_rows.append({"detector": n, "threshold": det.threshold, "clean_false_alarm_rate": float(det.predictions.mean()),
                       "per_run_min": per_run.min(), "per_run_median": per_run.median(), "per_run_max": per_run.max()})
clean_table = pd.DataFrame(clean_rows).set_index("detector")
print(f"target FPR for calibrated detectors: {TARGET_FPR}")
clean_table.round(4)

## 5. Synthetic anomalies, one family at a time (same rows and labels for every detector)
`prevalence` is the share of anomalous rows; `random_pr_auc` is the PR-AUC a random ranking would reach (= prevalence). Accuracy is deliberately not reported: with this prevalence a detector that never alarms already scores ≈ 1 − prevalence.

In [ ]:
gen = AnomalyGenerator().fit(train)
store, tables = {}, []
for family in FAMILIES:
    r = gen.generate(eval_x, AnomalySpec(family, ANOMALY_RATE), seed=SEED, groups=eval_meta["simulationRun"])
    dets = {n: d.detect(r.features) for n, d in DETECTORS.items()}
    store[family] = {"labels": r.labels.to_numpy().astype(bool), "dets": dets,
                     "sample": eval_meta["sample"].to_numpy(), "events": len(r.events)}
    t = compare_detectors(r.labels, dets)
    t.insert(0, "family", family)
    t["prevalence"] = prevalence(r.labels)
    t["random_pr_auc"] = t["prevalence"]
    tables.append(t)
    del r
comparison = pd.concat(tables).reset_index(names="detector")
view = comparison.set_index(["family", "detector"])
view[["precision", "recall", "f1", "false_positive_rate", "pr_auc", "roc_auc", "prevalence", "tp", "fp", "fn"]].round(4)

## 6. Uncertainty — bootstrap over whole runs
Every detector sees the same resampled runs, so the autoencoder-minus-PCA differences are paired. A difference whose 95 % interval contains 0 is **not** evidence that one detector is better.

In [ ]:
boot_tables, diff_rows = [], []
for family, s in store.items():
    detectors = {n: (d.scores, d.threshold) for n, d in s["dets"].items()}
    summary, reps = cluster_bootstrap(runs, s["labels"], detectors, n_boot=N_BOOT, seed=SEED)
    summary = summary.reset_index()
    summary.insert(0, "family", family)
    boot_tables.append(summary)
    for metric in ("recall", "roc_auc", "pr_auc"):
        for other in ("pca", "rule_check_equal_fpr"):
            d = paired_difference(reps, "autoencoder", other, metric)
            verdict = ("autoencoder higher" if d["ci_low"] > 0 else "autoencoder lower" if d["ci_high"] < 0 else "no clear difference")
            diff_rows.append({"family": family, "metric": metric, "autoencoder_minus": other, **d, "reading": verdict})
bootstrap = pd.concat(boot_tables, ignore_index=True)
differences = pd.DataFrame(diff_rows)
pivot = bootstrap[bootstrap["metric"].isin(["recall", "roc_auc", "pr_auc"])].copy()
pivot["estimate [95% CI]"] = pivot.apply(lambda r: f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1)
print(f"bootstrap: {N_BOOT} resamples of {len(eval_runs)} runs, seed {SEED}")
pivot.pivot_table(index=["family", "detector"], columns="metric", values="estimate [95% CI]", aggfunc="first")

In [ ]:
differences.round(4).set_index(["family", "metric", "autoencoder_minus"])

## 7. Sample-index effects
Training runs contain samples 1–500; testing runs 1–960. If a detector's alarm rate depends on the sample index, a threshold calibrated on pooled rows can be too loose or too tight at some indices. Clean-row alarm rates by band, on the calibration runs (used to set the thresholds) and on the evaluation runs:

In [ ]:
cal_meta = data.metadata["calibration"]
def band_table(frame_meta, x):
    cols = {}
    for n, d in DETECTORS.items():
        flags = d.detect(x, check_unseen=False).predictions
        cols[n] = rate_by_sample_band(flags, frame_meta["sample"].to_numpy(), SAMPLE_BAND_EDGES)["rate"]
    out = pd.DataFrame(cols)
    out.insert(0, "n_rows", rate_by_sample_band(np.zeros(len(x), bool), frame_meta["sample"].to_numpy(), SAMPLE_BAND_EDGES)["n_rows"])
    return out
cal_bands = band_table(cal_meta, data["calibration"])
eval_bands = band_table(eval_meta, eval_x)
print("CLEAN alarm rate by sample band - CALIBRATION runs (thresholds were set here):")
display(cal_bands.round(4))
print("CLEAN alarm rate by sample band - EVALUATION runs:")
display(eval_bands.round(4))

In [ ]:
# Detection rate (recall) of injected rows by sample band, pooled over the six families
rows = []
for n in DETECTORS:
    flags = np.concatenate([s["dets"][n].predictions[s["labels"]] for s in store.values()])
    samp = np.concatenate([s["sample"][s["labels"]] for s in store.values()])
    t = rate_by_sample_band(flags, samp, SAMPLE_BAND_EDGES)["rate"]
    rows.append(t.rename(n))
recall_bands = pd.DataFrame(rows).T
print("Share of INJECTED rows flagged, by sample band (pooled over families):")
recall_bands.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
for n in ("pca", "autoencoder"):
    sc = pd.Series(clean[n].scores).groupby(eval_meta["sample"].to_numpy()).mean()
    ax.plot(sc.index, sc.values / DETECTORS[n].threshold_, label=f"{n} (mean score / threshold)")
ax.axvline(500.5, color="k", ls="--", lw=0.8)
ax.set_xlabel("sample index within run (dashed = end of training-run length)"); ax.set_ylabel("score / threshold")
ax.set_title("Clean evaluation rows: score level vs sample index"); ax.legend()
fig.tight_layout(); plt.show()

## 8. Reading the results
Fill this in from the tables above (do not copy numbers from other notebooks):
* Which detectors reach their target false-alarm rate on clean evaluation rows?
* Per family, does the autoencoder exceed PCA or the equal-FPR rule check beyond the bootstrap interval?
* Is PR-AUC clearly above the random reference (the prevalence)?
* Does any detector's alarm rate depend on the sample index?
* Caveats: synthetic anomalies; one seed; one anomaly rate; evaluation runs were also visible during Phase 3 design (see `docs/RESEARCH_METHODOLOGY.md` §3.1).

In [ ]:
repro = {
    "seed": SEED, "target_fpr": TARGET_FPR, "anomaly_rate": ANOMALY_RATE, "n_boot": N_BOOT,
    "eval_runs_limit": EVAL_RUNS_LIMIT, "sample_band_edges": SAMPLE_BAND_EDGES,
    "roles": data.summary().reset_index().to_dict(orient="records"),
    "eval_run_ids_first_last": [eval_runs[0], eval_runs[-1]], "n_eval_runs": len(eval_runs), "n_eval_rows": int(len(eval_x)),
    "model_dir": MODEL_DIR, "model_sha256": json.loads((paths.root / MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))["model_sha256"],
    "manifest": verify_manifest(paths.root / MODEL_DIR),
    "saved_scaler_matches_workflow_scaler": bool(same_cols and d_center < 1e-9 and d_scale < 1e-9),
    "saved_threshold_minus_recomputed": float(ae.threshold_ - fresh),
    "holdout_loaded": False, "holdout_runs_previously_scored_excluded": list(HOLDOUT_RUNS_PREVIOUSLY_SCORED),
    "synthetic_anomalies_only": True,
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__},
}
try:
    import tensorflow as tf
    repro["versions"]["tensorflow"] = tf.__version__
except ImportError:
    pass
print(json.dumps({k: v for k, v in repro.items() if k != "roles"}, indent=2))

## 9. Save (new files only — the original `reports/evaluation_metrics.json` is never touched)

In [ ]:
if WRITE_REPORT:
    from production_pulse.reporting import write_evaluation_report
    from production_pulse.paths import safe_output_path

    ci = bootstrap.pivot_table(index=["family", "detector"], columns="metric", values=["ci_low", "ci_high"])
    ci.columns = [f"{m}_{k}" for k, m in ci.columns]
    records = comparison.merge(ci.reset_index(), on=["family", "detector"]).to_dict(orient="records")
    meta_file = paths.root / MODEL_DIR / "metadata.json"
    model_id = json.loads(meta_file.read_text(encoding="utf-8"))["model_sha256"][:12]
    metrics = {
        "setup": {"seed": SEED, "anomaly_rate": ANOMALY_RATE, "eval_runs": len(eval_runs), "eval_rows": int(len(eval_x)),
                  "target_fpr": TARGET_FPR, "n_boot": N_BOOT, "anomalies": "SYNTHETIC (injected); not real faults",
                  "operating_point": "all calibrated detectors at the same target FPR on calibration-role runs"},
        "threshold": ae.threshold_info(),
        "clean_false_alarm_rate": float(clean["autoencoder"].predictions.mean()),
        "detector_comparison": records,
        "clean_false_alarm_by_detector": clean_table.reset_index().to_dict(orient="records"),
        "autoencoder_minus_baselines": differences.to_dict(orient="records"),
        "sample_index_effects": {"clean_calibration_runs": cal_bands.reset_index().to_dict(orient="records"),
                                 "clean_evaluation_runs": eval_bands.reset_index().to_dict(orient="records"),
                                 "injected_rows_flagged": recall_bands.reset_index().to_dict(orient="records")},
        "reproducibility": repro,
    }
    out = write_evaluation_report(metrics, paths, source="notebooks/04_equal_fpr_comparison.ipynb",
                                  model_id=model_id, target=REPORT_TARGET, overwrite=OVERWRITE)
    print("saved", out.relative_to(paths.root))
    for name, table in {f"equal_fpr_comparison{RUN_SUFFIX}.csv": comparison, f"equal_fpr_bootstrap{RUN_SUFFIX}.csv": bootstrap,
                        f"equal_fpr_differences{RUN_SUFFIX}.csv": differences}.items():
        target = safe_output_path(paths.reports_dir / name, paths, overwrite=OVERWRITE)
        table.to_csv(target, index=False)
        print("saved", target.relative_to(paths.root))
else:
    print("WRITE_REPORT is False - nothing written.")